In [1]:
import multiprocessing
import psutil
import platform

def analyze_system():
    """Get detailed system information"""
    print("=== CPU Information ===")
    print(f"Physical cores: {psutil.cpu_count(logical=False)}")
    print(f"Logical cores (with hyperthreading): {psutil.cpu_count(logical=True)}")
    print(f"CPU frequency: {psutil.cpu_freq().current:.2f} MHz")
    print(f"Architecture: {platform.machine()}")
    
    print("\n=== Memory Information ===")
    mem = psutil.virtual_memory()
    print(f"Total RAM: {mem.total / (1024**3):.2f} GB")
    print(f"Available RAM: {mem.available / (1024**3):.2f} GB")
    print(f"Used RAM: {mem.used / (1024**3):.2f} GB")
    print(f"Memory usage: {mem.percent}%")
    
    print("\n=== Recommended Settings ===")
    # Leave 1-2 cores for system
    recommended_workers = max(1, psutil.cpu_count(logical=False) - 1)
    print(f"Recommended worker processes: {recommended_workers}")
    
    # Estimate memory per worker
    usable_memory = mem.available * 0.8  # Use 80% of available
    memory_per_worker = usable_memory / recommended_workers
    print(f"Memory per worker: {memory_per_worker / (1024**3):.2f} GB")
    
    return {
        'physical_cores': psutil.cpu_count(logical=False),
        'logical_cores': psutil.cpu_count(logical=True),
        'total_memory_gb': mem.total / (1024**3),
        'available_memory_gb': mem.available / (1024**3),
        'recommended_workers': recommended_workers
    }

# Run this to understand your system
system_info = analyze_system()

=== CPU Information ===
Physical cores: 64
Logical cores (with hyperthreading): 64
CPU frequency: 3868.07 MHz
Architecture: x86_64

=== Memory Information ===
Total RAM: 755.45 GB
Available RAM: 596.74 GB
Used RAM: 153.86 GB
Memory usage: 21.0%

=== Recommended Settings ===
Recommended worker processes: 63
Memory per worker: 7.58 GB


In [4]:
import numpy as np
import sys

def estimate_memory_usage(L, num_chunks_1, num_chunks_2):
    """
    Estimate memory usage for chunked FlexDTW.
    
    Parameters:
    -----------
    L : int
        Chunk size (e.g., 1000)
    num_chunks_1, num_chunks_2 : int
        Number of chunks in each dimension
    """
    
    print("=== Memory Estimation ===\n")
    
    # 1. Input cost matrix C (if kept in memory)
    total_size_1 = num_chunks_1 * (L - 1) + 1
    total_size_2 = num_chunks_2 * (L - 1) + 1
    C_memory = total_size_1 * total_size_2 * 8  # float64
    print(f"Cost matrix C: {total_size_1} × {total_size_2}")
    print(f"  Memory: {C_memory / (1024**3):.2f} GB\n")
    
    # 2. Per-chunk storage
    print(f"Per chunk ({L}×{L}):")
    chunk_arrays = {
        'C': L * L * 8,      # Cost matrix
        'D': L * L * 8,      # Cumulative cost
        'S': L * L * 8,      # Starting points
        'B': L * L * 8,      # Backpointers
    }
    
    per_chunk_total = sum(chunk_arrays.values())
    for name, size in chunk_arrays.items():
        print(f"  {name}: {size / (1024**2):.2f} MB")
    print(f"  Total per chunk: {per_chunk_total / (1024**2):.2f} MB\n")
    
    # 3. All chunks storage
    num_chunks = num_chunks_1 * num_chunks_2
    all_chunks_memory = per_chunk_total * num_chunks
    print(f"All {num_chunks} chunks:")
    print(f"  Memory: {all_chunks_memory / (1024**3):.2f} GB\n")
    
    # 4. Edge storage (D_chunks, L_chunks)
    # Each chunk has 2 edges (top and right), each edge has ~L positions
    edge_storage_per_chunk = 2 * L * 8 * 2  # 2 edges, L positions, 8 bytes, 2 arrays (D and L)
    edge_storage_total = edge_storage_per_chunk * num_chunks
    print(f"Edge storage (D_chunks, L_chunks):")
    print(f"  Per chunk: {edge_storage_per_chunk / (1024**2):.2f} MB")
    print(f"  Total: {edge_storage_total / (1024**3):.2f} GB\n")
    
    # 5. Sparse starting points (very small)
    # Estimate: ~10% of edge positions are valid (very conservative)
    sparse_storage = num_chunks * 2 * L * 0.1 * 8  # 10% sparsity, 8 bytes per int
    print(f"Sparse starting points (~10% sparsity):")
    print(f"  Memory: {sparse_storage / (1024**2):.2f} MB\n")
    
    # 6. Total memory
    total_memory = C_memory + all_chunks_memory + edge_storage_total + sparse_storage
    print(f"=== TOTAL MEMORY ===")
    print(f"  {total_memory / (1024**3):.2f} GB")
    print(f"  Peak usage (with overhead): ~{total_memory * 1.5 / (1024**3):.2f} GB\n")
    
    # 7. Multiprocessing considerations
    print("=== Multiprocessing Considerations ===")
    print("When using multiprocessing.Pool:")
    print(f"  - Each worker needs chunk data (~{per_chunk_total / (1024**2):.2f} MB per chunk)")
    print(f"  - Data is copied to worker processes (no shared memory by default)")
    print(f"  - For N workers processing M chunks simultaneously:")
    print(f"    Memory ≈ Base + N × M × {per_chunk_total / (1024**2):.2f} MB")
    
    return {
        'total_memory_gb': total_memory / (1024**3),
        'per_chunk_mb': per_chunk_total / (1024**2),
        'edge_storage_gb': edge_storage_total / (1024**3)
    }

# Example usage
memory_info = estimate_memory_usage(L=1000, num_chunks_1=50, num_chunks_2=50)
# ```

### **Memory Bottlenecks in Our Algorithm**

# 1. **Chunk Storage**: Each chunk stores C, D, S, B matrices (4 × L² floats)
# 2. **Edge Storage**: D_chunks and L_chunks (relatively small, 2 × L per chunk)
# 3. **Multiprocessing Overhead**: Data copied to worker processes
# 4. **Operating System**: Needs free memory for other processes

# ### **Rule of Thumb for Memory Constraints**
# ```
# Available Memory ≥ Total Data Size × 1.5 × (1 + Number of Workers × 0.2)

=== Memory Estimation ===

Cost matrix C: 49951 × 49951
  Memory: 18.59 GB

Per chunk (1000×1000):
  C: 7.63 MB
  D: 7.63 MB
  S: 7.63 MB
  B: 7.63 MB
  Total per chunk: 30.52 MB

All 2500 chunks:
  Memory: 74.51 GB

Edge storage (D_chunks, L_chunks):
  Per chunk: 0.03 MB
  Total: 0.07 GB

Sparse starting points (~10% sparsity):
  Memory: 3.81 MB

=== TOTAL MEMORY ===
  93.17 GB
  Peak usage (with overhead): ~139.76 GB

=== Multiprocessing Considerations ===
When using multiprocessing.Pool:
  - Each worker needs chunk data (~30.52 MB per chunk)
  - Data is copied to worker processes (no shared memory by default)
  - For N workers processing M chunks simultaneously:
    Memory ≈ Base + N × M × 30.52 MB
